# 署名を作成しよう

- データに署名を行うことでデータの改竄を防ぐことができます。

- ハイブリッド暗号の仕組みを見てみよう
---

### ・セクション1: 関数の準備

In [1]:
#ライブラリのインストール
%pip install pycryptodome


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.Hash import SHA256
from Crypto.Signature import pss
import os

In [3]:
#公開鍵の生成
def generate_keys():
    key = RSA.generate(2048)
    private_key = key
    public_key = key.publickey()
    return private_key, public_key

In [4]:
#暗号化関数
def do_encrypt(message, public_key):
    #1.共通鍵（セッションキー）の生成
    session_key = os.urandom(32)
    
    #2.共通鍵を送信相手の公開鍵で暗号化
    cipher_rsa = PKCS1_OAEP.new(public_key, hashAlgo=SHA256)
    encrypted_key = cipher_rsa.encrypt(session_key)
    
    #3.共通鍵でメッセージを暗号化
    cipher_aes = AES.new(session_key, AES.MODE_GCM)
    nonce = cipher_aes.nonce
    
    #4.暗号化と同時に認証タグを生成
    c, tag = cipher_aes.encrypt_and_digest(message)
    
    #暗号化された共通鍵、nonce、署名、暗号文を結合して返す
    contents = [encrypted_key, nonce, tag, c]
    return contents

In [5]:
#復号関数
def do_decrypt(contents, private_key):
    encrypted_key = contents[0]
    nonce = contents[1]
    tag = contents[2]
    chiphertext = contents[3]
    
    #1.秘密鍵でセッションキーを復号
    cipher_rsa = PKCS1_OAEP.new(private_key, hashAlgo=SHA256)
    session_key = cipher_rsa.decrypt(encrypted_key)
    
    #2.共通鍵でメッセージを復号
    cipher_aes = AES.new(session_key, AES.MODE_GCM, nonce=nonce)
    
    #3.復号と同時に認証タグの検証を行う
    try:
        m = cipher_aes.decrypt_and_verify(chiphertext, tag)
        return m
    
    except ValueError:
        raise ValueError("メッセージまたはタグが改ざんされています (認証失敗)")

In [6]:
#署名の作成(送信側)
def makesign(message, private_key):
    h = SHA256.new(message)
    signer = pss.new(private_key)
    signature = signer.sign(h)
    return signature

#署名の検証(受信側)
def verification(message, signature, public_key):
    h = SHA256.new(message)
    verifier = pss.new(public_key)
    
    try:
        verifier.verify(h, signature)
        return True
    except (ValueError, TypeError):
        return False

In [7]:
#s_sk = 送信者の秘密鍵
#s_pk = 送信者の公開鍵
#r_sk = 相手の秘密鍵
#r_pk = 相手の公開鍵

#送信関数
def send(s_sk, r_pk):
    #メッセージ作成
    message = input("メッセージを入力: ").encode('utf-8')

    #デジタル署名
    sign= makesign(message, s_sk)

    #ハイブリッド暗号化
    contents= do_encrypt(message, r_pk)

    #送信するデータをリストで出力
    data = [contents, sign]
    print("データの暗号化と署名作成に成功")
    return data

#受信関数
def recieve(data, r_sk, s_pk):
    contents = data[0]
    sign = data[1]
    try:
        decrypted_message = do_decrypt(contents, r_sk)
        
        #デジタル署名の検証
        is_valid = verification(decrypted_message, sign, s_pk)
        print(f"署名の検証結果: {is_valid}")
    
        if is_valid:
            print(f"復号されたメッセージ: 「{decrypted_message.decode()}」")
        else:
            print("署名が不正です。送信者が異なるか、メッセージが改ざんされています。")
        
    except ValueError as e:
        print(f"復号失敗: {e}")

---

### ・セクション３: メッセージを送るテストをしてみよう

In [8]:
#鍵ペアの生成
#教員の署名用
T_sk, T_pk = generate_keys()
#生徒の暗号化用
S_sk, S_pk = generate_keys()
print("鍵ペアが生成されました。\n")

鍵ペアが生成されました。



In [11]:
"""
演習課題(1) 関数に適切な鍵を入れてテストを行なってみよう
"""
data = send()

メッセージを入力:  t


データの暗号化と署名作成に成功


In [12]:
#どのように暗号化されたのか確認してみよう。
print("-"*15, "セッションキー, nonce, タグ, 暗号文の順に出力")
for d in data[0]:
    print(d)
    print("\n")

print("-"*15, "署名")
print(data[1])
print("\n")

--------------- セッションキー, nonce, タグ, 暗号文の順に出力
b'AC\x07%t<S\xb8\xbe ;z\x1e\x81\r\x01\xa7\xd6\xc5\xe8y=\xba\x97\x9d\xc8\xd6XVH\xb8_J$\xd4.\xb3\xe5\x13-a\x14\x1e\xf4_\xfc\xb4Cq\xfbv\xab4*\x17i\xc8\xbe#\x91;|\xa7\xc6\xf5\xd3%V\xdfk\xd6\xe2\x06eY\xfa#CKM\xd9\x9b_\xe8\xbc\x96\x85\xf4\xa43bp\x9b\xcd\xbd\x81\xe1\x9b\xf1\xa8k\x98\'\xb1>\x9a\xb9\x81\x0f"\x18_\x96\xff\xf1\x03#6\x82\x0et\x8d\x85\xf8E\x1eM\xfb\x93\xa9c\xe0\x16y\x826W\xba\xcd\xb2\x86\x8f\x19\xf1\x1b} X)\x18\x05\xc7\xac\x8b\xf5}\x87U\x03G\x10\xceZ\x8a_ehm\xac\x12uV\x99\xe1\xc0\xf4\x025\x11\x92\x07\xa0\x85u\xcduY\xf4\xa9t\r[\x8a\xf6/\xb9\xd8\x8e\xc7"`viFJ\x02\xabb_\x9d\xf9\r;\x9d\xaf\x1f\xf0jY\xac\xdc\x11\xcc\xe9\x0c\xff\\\x99\xc9\xd0\xe6xrBX\xf74\xfb3\x1dA\xcf\x8a\x9b\xb8hA.d\xe3\xc6\xde\x0c\xfaH\xfb'


b'\x0e\x06=\x070;\xda\x1d\xf2$|\n8%{\xfc'


b'\x0bz\x1d\xcc\x91\xdf\xed\xa3i\x8e\x96OD]\xe2\xff'


b'\xcb'


--------------- 署名
b"\x95\xd6@\x0br\xd9\xf1f\xab\x9f\xb3b,\xb1\x17\xe4\x19\x1e\xa3%\x9f\x08\xe8\xcb\xa2\x86\t\xf8Z\x90\x0f\xdc

In [13]:
#以下に正しい引数を記入して押して正しくメッセージが送信されたか確認
recieve()

署名の検証結果: True
復号されたメッセージ: 「t」


---
### ・section4: 実際に隣の人と秘匿通信をしてみよう
   (実験してないうまくいくかわかりません)

In [14]:
#鍵の中身をコピペできる形に変更
#出力をコピペして、メモにし、エアドロで送信
public_pem = S_pk.export_key(format='PEM')
print(public_pem.decode())

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA4R32/0UkkOu9nYx61cin
1eULeUqWcNMC/gPEPBNDMnFyp8IRS62BNh8Osrt4uxf5/YgJSz2SFfq6eDSBcEqO
R2maiVao2eB9Z7x6yLcxHMT0psyxmgWY0M5CSUFqUZ01Ien4LUNsW071oUbtl/3b
MOr9fq9HBpf0K4TNt2O/zJM9orUc8Ox8C5JhLlOhrqnuedN6kx7UNFizGegzleg8
mmJ2AZCc5lvs93KJ0P4wmhCvK1KHokmR+w1QZpmX/IzMcXBaz4XCZiCfcQQ72sKV
CsXj0q3mY0CTQ+PJyY6bQ3szUyV1vF0RMoqwfAzwHIEFMUJNROiJXQGgdeFPnpux
bQIDAQAB
-----END PUBLIC KEY-----


In [15]:
#隣の人の公開鍵をもらい、元の形に復元
copied_public_pem = """
"""
n_pk = RSA.import_key(copied_public_pem.encode('utf-8'))

In [16]:
#隣の人に出力結果をメモに保存し、エアドロで送信
data = send(S_sk, n_pk)
contents = data[0]
sign = data[1]
print("encrypted_key=", contents[0])
print("nonce=", contents[1])
print("tag=", contents[2])
print("c=", contents[3])
print("sign=", sign)

メッセージを入力:  t


データの暗号化と署名作成に成功
encrypted_key= b'.T"\xa7\xea:\x1ai\x8c\x15\xe0\x18d\x1a\xc4np\xac\x84-\xe0\xb5\xec\x10\x91\x9dw\xd2`\xa5\xfe\x98\xb9\xdd\xc0\xce\xc3\xcf\x97\x88Z\xd3Z\xcb[\xd4T\xe2>\x1d\xb6\x7f\xdf\xec\xba@\x88\xfa\x96\x127x\xe8An\t\xbf\xeb\x07\xc1\x95\xcc\x8d*gE\xff[\xe0\x13SNb)\xa5\xf7\xf3\xa0\xc5=\xbff\x19\xcb\x88m\x19?\xfe\x97{\x9d\xb3_\x06\xed\xe7\xdb:%\xa95M\x96X\xcd2\x00\xb6\x80\xd5\xf8\xd1\xf7q\x17\xf7\x02\xc9==\x9e\xd4\r\x07\x10\xc9\xa1SyH\xdfzx\x10\rj\x1f\xdb\xac\xcb\xf2\xab\xe3\xda\xcd|\xf0h\xe5\xaf\xcd\xef\xb5\xcf\xf5\x85\x91\xdcL\xcbj\xe7\xca\x10\xb8H\xe7\xdf\xf1H\xc6\xea\x9cD8dE\xdb\xf8\x9f\xe5\xbb\xe1c\xcf\xa9\xe2\xcf\xfdr\x8a\xd0\xf5\x1bxa\xc4\x82\x81\xac>\xf2Pw\x97@\xc3p\xffh\xf8\x81\xaa\x14\'\x99\xe8\x81d(\x0cj\xf1\x9f\x89\xddR\x81\xd1/`L\xeb\xf4\xe5C\xd1\x8fw=\r\xe4"\x89\x8d'
nonce= b'\xb4?lS?\xf0b\xa6"\x84\xee27;\xde\xcb'
tag= b'\x8e\x80\x81\x92\xdd\xeb\x02\x0c\x13\xe8\x11K\xb9\xf4u\xbb'
c= b'/'
sign= b'.K.\xd3\x93\xfe\xb7@\xf0^&-5\xcd\xa3\x0e\xe4\x02t\xcc\xce\xb2\x

In [ ]:
#隣の人のメモの結果をコピペ

In [ ]:
#隣の人の結果
contents = [encrypted_key, nonce, tag, c]
data = [contens, sign]
recieve(data, S_sk, n_pk)

---
### ・セクション5: メッセージの改ざんに対策ができるか確認しよう

In [ ]:
#上のコードのc=の部分を以下の先生が改ざんしたメッセージに変更して改ざんが検出できるか確かめる
c = 